# Brain tumour segmentation + classification (Colab)

Runs `train_tumor.py` (PyTorch): one multi-task U-Net that outlines the tumour and predicts its type (meningioma / glioma / pituitary).

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells in order.

Results are saved to your Google Drive, so they survive if Colab disconnects.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Mount Google Drive (for saving results)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/brain-tumor-multitask/models"
import os; os.makedirs(OUT, exist_ok=True)
print("Results will be saved to", OUT)

## 3. Get the code

The repo is **private**, so Colab needs a GitHub token:

1. GitHub → Settings → Developer settings → **Fine-grained tokens** → Generate. Give it **read-only Contents** access to `brain-tumor-multitask` only.
2. In Colab, click the **key icon (Secrets)** on the left, add a secret named `GITHUB_TOKEN` with the token, and turn on **Notebook access**.

(If you make the repo public later, this cell works without a token.)

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = (f"https://{token}@github.com/ChanduVenkatSaiNarne/brain-tumor-multitask.git" if token
       else "https://github.com/ChanduVenkatSaiNarne/brain-tumor-multitask.git")
if not os.path.exists("/content/brain-tumor-multitask"):
    r = subprocess.run(["git", "clone", "-q", url, "/content/brain-tumor-multitask"], capture_output=True, text=True)
    print(r.stderr.replace(token, "***") if token else r.stderr or "Cloned.")
%cd /content/brain-tumor-multitask
!git log --oneline -1
!pip -q install -r requirements.txt

## 4. Get the data (Figshare, ~900 MB)

**Option A (default):** download directly from Figshare into Colab.

**Option B:** if the download fails, zip your `Downloads\1512427` folder on your laptop, upload it to Google Drive as `MyDrive/brain_tumor_data.zip`, and set `USE_DRIVE_ZIP = True`.

In [ ]:
import glob, os
USE_DRIVE_ZIP = False
DATA = "/content/data/figshare"
os.makedirs(DATA, exist_ok=True)

if USE_DRIVE_ZIP:
    !unzip -q -o /content/drive/MyDrive/brain_tumor_data.zip -d {DATA}
else:
    !wget -q --show-progress -O /content/figshare.zip "https://figshare.com/ndownloader/articles/1512427"
    !unzip -q -o /content/figshare.zip -d {DATA}

# The Figshare download contains four inner zip files; unzip those too
for z in glob.glob(f"{DATA}/**/*.zip", recursive=True):
    !unzip -q -o "{z}" -d "{os.path.dirname(z)}"

mats = [p for p in glob.glob(f"{DATA}/**/*.mat", recursive=True) if not p.endswith("cvind.mat")]
print(len(mats), "image .mat files found (expected 3064)")

## 5. Quick test (2 epochs, a few minutes)
Checks that everything works before the long run.

In [ ]:
!python train_tumor.py --figshare-dir {DATA} --out /content/test_run --epochs 2

## 6. Full training
Up to 150 epochs with early stopping, roughly 30–90 minutes on a T4. Keep this tab open.

In [ ]:
!python train_tumor.py --figshare-dir {DATA} --out {OUT}

## 7. Test results

In [ ]:
import json
r = json.load(open(f"{OUT}/test_results_tumor.json"))
print(f"Test patients: {r['n_test_patients']}  |  test slices: {r['n_test_slices']}")
print(f"Dice (mask):   {r['test_dice_mean']:.3f} ± {r['test_dice_std']:.3f}")
for k, v in r["test_dice_per_class"].items():
    print(f"   {k:<11} {v:.3f}")
print(f"Accuracy:      {r['test_accuracy']:.3f}")
print(f"Macro-F1:      {r['test_macro_f1']:.3f}")
print("Confusion matrix (rows = true, cols = predicted):", r["confusion_matrix"]["classes"])
for row in r["confusion_matrix"]["matrix"]:
    print("   ", row)

## 8. Training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv(f"{OUT}/history_tumor.csv")
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(h["loss"], label="train"); ax[0].plot(h["val_loss"], label="val"); ax[0].set_title("Total loss")
dice_cols = [c for c in h.columns if "dice" in c]
acc_cols = [c for c in h.columns if "accuracy" in c]
for c in dice_cols: ax[1].plot(h[c], label=c)
for c in acc_cols: ax[2].plot(h[c], label=c)
ax[1].set_title("Mask Dice"); ax[2].set_title("Type accuracy")
for a in ax: a.set_xlabel("epoch"); a.legend()
plt.show()

## 9. Example predictions on TEST patients

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from data_utils import load_figshare, patient_split
from model import MultiTaskUNet, predict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(f"{OUT}/unet_tumor_multitask_best.pt", map_location=device)
classes = ckpt["classes"]
model = MultiTaskUNet(n_classes=len(classes)).to(device)
model.load_state_dict(ckpt["model"])

x, m, y, pids, _ = load_figshare(DATA)
_, _, te = patient_split(pids)            # same split as training (seed 42)
pick = np.random.default_rng(0).choice(te, 6, replace=False)
pm, pt = predict(model, x[pick], device)

fig, ax = plt.subplots(2, 6, figsize=(20, 7))
for j, i in enumerate(pick):
    ax[0, j].imshow(x[i], cmap="gray"); ax[0, j].contour(m[i], colors="lime", linewidths=1)
    ax[0, j].set_title(f"True: {classes[y[i]]}", fontsize=10)
    ax[1, j].imshow(x[i], cmap="gray"); ax[1, j].contour(pm[j] > 0.5, colors="red", linewidths=1)
    ax[1, j].set_title(f"Pred: {classes[pt[j].argmax()]} ({pt[j].max():.0%})", fontsize=10)
    ax[0, j].axis("off"); ax[1, j].axis("off")
fig.suptitle("Top: ground truth (green)  |  Bottom: prediction (red)  - research use only")
plt.tight_layout(); plt.show()